<a href="https://colab.research.google.com/github/riza93n-hub/data-science-2026/blob/main/Pertemuan3_RIZA_250401020014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aktivitas Hands On Pertemuan 3
**Nama Mahasiswa:** RIZA  
**NIM:** 250401020014  
**Kelas:** IF401

In [1]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# Membaca file data kotor yang sudah diunggah
df = pd.read_csv('housing_dirty.csv')
print('Shape awal:', df.shape)
print('\nJumlah data kosong pada tiap kolom:')
print(df.isnull().sum())

Shape awal: (130, 7)

Jumlah data kosong pada tiap kolom:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [2]:
# STEP 1 — Hapus Baris yang Duplikat
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

# STEP 2 — Normalisasi Huruf dan Teks (Menghapus spasi luar & merapikan huruf kapital)
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# STEP 3 — Mengisi Nilai Kosong (Imputasi)
# Mengisi angka kosong dengan nilai tengah (median)
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
# Mengisi kolom kamar yang kosong dengan nilai yang paling sering muncul (modus)
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print('\nHasil pengecekan data kosong setelah diisi:')
print(df.isnull().sum())

Setelah hapus duplikat: (130, 7)

Hasil pengecekan data kosong setelah diisi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [3]:
# STEP 4 — Menangani Data Ekstrem / Batas Wajar (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    if col in df.columns:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        # Membatasi nilai agar tidak melewati batas bawah dan batas atas wajar
        df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# STEP 5 — Validasi & Ekspor Hasil
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

print('Shape akhir dataset bersih:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih berhasil disimpan dengan nama: housing_clean.csv')

Shape akhir dataset bersih: (130, 7)
Dataset bersih berhasil disimpan dengan nama: housing_clean.csv


## Kesimpulan

* **Apa yang Dipelajari:** Di pertemuan ketiga ini, saya belajar cara membersihkan data yang kotor atau berantakan sebelum dipakai untuk analisis lebih lanjut. Saya mempraktikkan cara membuang baris data yang ganda (duplikat), menyamakan format penulisan teks tabel agar rapi dan seragam, mengisi bagian data yang bolong atau kosong menggunakan nilai tengah atau nilai yang paling sering muncul, serta membatasi nilai-nilai ekstrem yang terlalu jauh dari kewajaran.
* **Temuan Utama:** Data mentah di dunia nyata sering kali memiliki banyak masalah, mulai dari spasi teks yang berantakan, data ganda yang tidak sengaja terinput, hingga nilai kosong. Melalui proses pembersihan ini, kita bisa memastikan kualitas data menjadi jauh lebih baik tanpa harus menghapus seluruh baris data. Pembatasan nilai ekstrem menggunakan metode IQR juga sangat berguna agar nilai-nilai yang terlalu besar atau kecil tidak merusak keakuratan model analisis nantinya.
* **Keterbatasan / Pertanyaan yang Muncul:** * *Keterbatasan:* Cara mengisi data kosong dengan nilai tengah atau nilai terbanyak di atas langsung diterapkan ke seluruh baris data tanpa melihat kelompok kategori lainnya (seperti tipe rumah atau lokasi kota asal), sehingga nilainya bisa jadi kurang presisi untuk rumah di kota tertentu.
  * *Pertanyaan:* Bagaimana cara mengisi data kosong secara lebih pintar, misalnya mengisi luas rumah yang hilang berdasarkan rata-rata luas rumah di kota atau wilayah yang sama saja, bukan digeneralisir dari keseluruhan isi tabel?